In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['special.jpg', 'IMG_20200327_184446.jpg', 'IMG_20191019_123403 (1).jpg', 'IMG_20191019_123403.jpg', 'IMG_20191220_081157.jpg', 'Mansi gift.png', 'IMG_1051.heic', 'IMG_1053.heic', 'Colab Notebooks', 'Nanu', 'Alkali Metals.pdf', 'Photos', 'Solutions Chemistry.gdoc', 'Nepal trip.gdoc', 'Current electricity.gdoc', 'Moving charges and magnetism.gdoc', 'Magnetism and matter.gdoc', 'Untitled presentation (2).gslides', 'Study-Gudiya-Mukunda', 'code.gdoc', 'ddb-cqwx-qmr - Oct 28, 2022.pdf', 'Untitled document (1).gdoc', 'PICS.gdoc', 'Untitled form.gform', 'Space game:.gdoc', 'Soumistha-Bhuyan-12-Marksheet (2).pdf', 'Soumistha-Bhuyan-12-Marksheet (1).pdf', 'Soumistha-Bhuyan-12-Marksheet.pdf', 'HallTicket-CBSE-Class-XII (1).pdf', 'HallTicket-CBSE-Class-XII.pdf', 'family photo.jpg', 'VIT', 'pictures', 'Module 1.gdoc', 'plots.gdoc', '203bbcf5-f1f7-411d-b6ea-7ec0f001ab26.jpeg', 'frens (Responses).gsheet', 'frens.gform', '389caa3d-ee80-48a3-b1ca-373f1b9e02a8.jpeg', 'Untitled document.gdoc', 'Untitle

In [ ]:

import cv2
import os

base_path = "/content/drive/MyDrive/fall_detection_dataset"
frames_folder = base_path + "/frames"

os.makedirs(frames_folder, exist_ok=True)

video_files = [f for f in os.listdir(base_path) if f.endswith(".mp4")]

frame_count = 0

for video in video_files:
    video_path = os.path.join(base_path, video)
    cap = cv2.VideoCapture(video_path)

    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # save every 20th frame (IMPORTANT: reduces workload)
        if count % 20 == 0:
            frame_name = f"{video}_frame_{frame_count}.jpg"
            cv2.imwrite(os.path.join(frames_folder, frame_name), frame)
            frame_count += 1

        count += 1

    cap.release()

print("✅ All frames extracted!")

✅ All frames extracted!


In [ ]:
!pip install ultralytics

from ultralytics import YOLO
import os

base_path = "/content/drive/MyDrive/fall_detection_dataset"

image_folder = base_path + "/images/train"
label_folder = base_path + "/labels/train"

os.makedirs(label_folder, exist_ok=True)

model = YOLO("yolov8n.pt")

CLASS_ID = 1  # temporarily all NORMAL

for img_name in os.listdir(image_folder):

    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(image_folder, img_name)
    results = model(img_path)[0]

    h, w = results.orig_shape

    label_path = os.path.join(label_folder, img_name.replace(".jpg", ".txt"))

    with open(label_path, "w") as f:
        for box in results.boxes:
            if int(box.cls[0]) == 0:  # person
                x1, y1, x2, y2 = box.xyxy[0]

                x_center = ((x1 + x2) / 2) / w
                y_center = ((y1 + y2) / 2) / h
                width = (x2 - x1) / w
                height = (y2 - y1) / h

                f.write(f"{CLASS_ID} {x_center} {y_center} {width} {height}\n")

print("Labels created (all normal)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

image 1/1 /content/drive/MyDrive/fall_detection_dataset/images/train/WhatsApp Video 2026-03-05 at 9.08.28 PM.mp4_frame_0.jpg: 608x640 2 persons, 43.5ms
Speed: 5.0ms preprocess, 43.5ms inference, 48.2ms postprocess per image at shape (1, 3, 608, 640)

image 1/1 /content/drive/MyDrive/fall_detection_dataset/images/train/WhatsApp Video 2026-03-05 at 9.08.28 PM.mp4_frame_3.jpg: 608x640 3 persons, 7.7ms
Speed: 2.0ms preprocess, 7.7ms inference, 1.3ms postprocess per image at shape (1, 3, 608, 640)

image 1/1 /content/drive/MyDrive/fall_detection_dataset/images/train/WhatsApp Video 2026-03-05 at 9.08.28 PM.mp4

In [ ]:
print(os.listdir(label_folder)[:5])

['WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_213.txt', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_215.txt', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_216.txt', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_214.txt', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_217.txt']


In [ ]:
import os

base_path = "/content/drive/MyDrive/fall_detection_dataset"

label_folder = base_path + "/labels/train"
normal_folder = base_path + "/normal_images"

# Check folder
if not os.path.exists(normal_folder):
    print("❌ normal_images folder NOT found!")
else:
    print("✅ normal_images folder found")

# Load normal images
normal_images = set(os.listdir(normal_folder))

print("Sample normal images:", list(normal_images)[:5])

match_count = 0

for file in os.listdir(label_folder):

    if not file.endswith(".txt"):
        continue

    image_name = file.replace(".txt", ".jpg")
    path = os.path.join(label_folder, file)

    with open(path, "r") as f:
        lines = f.readlines()

    with open(path, "w") as f:
        for line in lines:
            parts = line.split()

            # ✅ CORRECT MATCHING
            if image_name in normal_images:
                parts[0] = "1"
                match_count += 1   # 🔥 important
            else:
                parts[0] = "0"

            f.write(" ".join(parts) + "\n")

print("✅ Labels updated")
print("Matched NORMAL labels:", match_count)

✅ normal_images folder found
Sample normal images: ['YTDown.com_Shorts_Fall-detection_Media_EmLSgfpuv4I_001_720p.mp4_frame_344.jpg', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_145.jpg', 'Screen Recording 2026-03-23 213115.mp4_frame_388.jpg', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_185.jpg', 'WhatsApp Video 2026-03-05 at 9.16.13 PM.mp4_frame_149.jpg']
✅ Labels updated
Matched NORMAL labels: 391


In [ ]:
import os

label_folder = "/content/drive/MyDrive/fall_detection_dataset/labels/train"

count_0 = 0
count_1 = 0

for file in os.listdir(label_folder):

    if not file.endswith(".txt"):
        continue

    with open(os.path.join(label_folder, file)) as f:
        for line in f:
            if line.startswith("0"):
                count_0 += 1
            elif line.startswith("1"):
                count_1 += 1

print("Fall:", count_0)
print("Normal:", count_1)

Fall: 622
Normal: 391


In [ ]:
import os
import shutil

base_path = "/content/drive/MyDrive/fall_detection_dataset"

images_train = base_path + "/images/train"
labels_train = base_path + "/labels/train"

os.makedirs(images_train, exist_ok=True)
os.makedirs(labels_train, exist_ok=True)

In [ ]:
print("Images:", len(os.listdir(images_train)))
print("Labels:", len(os.listdir(labels_train)))

Images: 427
Labels: 427


In [ ]:
 data = f"""
path: {base_path}
train: images/train
val: images/train

names:
  0: fall
  1: normal
"""

with open("data.yaml", "w") as f:
    f.write(data)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="data.yaml",
    epochs=40,
    imgsz=640,   #image resolution
    batch = 8,
    lr0 = 0.001
)

#params:
#box_loss: how wrong is the box loaction
#cls_loss: classification loss: wrong class implies high loss
#dfl_loss: distributional focal loss: how preciselt box edges cover object
#instances: how many objects yolo sees per batch

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, po

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b95cc2e1730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import glob
print(glob.glob("/content/runs/detect/*/weights/best.pt"))

['/content/runs/detect/train/weights/best.pt']


In [ ]:
import shutil
import glob

path = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))[-1]

shutil.copy(
    path,
    "/content/drive/MyDrive/fall_detection_dataset/best.pt"
)

print("✅ Saved from:", path)

✅ Saved from: /content/runs/detect/train/weights/best.pt


In [ ]:
model = YOLO("/content/drive/MyDrive/fall_detection_dataset/best.pt")
model.train(
    data="data.yaml",
    epochs=20,
    imgsz=960,
    batch = 4
)

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/fall_detection_dataset/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b95880d0140>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import glob
print(glob.glob("/content/runs/detect/*/weights/best.pt"))

['/content/runs/detect/train2/weights/best.pt', '/content/runs/detect/train/weights/best.pt']


In [ ]:
import shutil
import glob

path = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))[-1]

shutil.copy(
    path,
    "/content/drive/MyDrive/fall_detection_dataset/best.pt"
)

print("✅ Saved from:", path)

✅ Saved from: /content/runs/detect/train2/weights/best.pt


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/fall_detection_dataset/best.pt")

In [ ]:
import os
import cv2
from ultralytics import YOLO

# Load trained model
model = YOLO("/content/drive/MyDrive/fall_detection_dataset/best.pt")

input_folder = "/content/drive/MyDrive/fall_detection_dataset/images/train"
output_folder = "/content/drive/MyDrive/fall_detection_dataset/output_images"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):

    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(input_folder, img_name)
    img = cv2.imread(img_path)

    results = model.predict(img, conf=0.25, iou = 0.5)[0]

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        height = y2 - y1
        width = x2 - x1
        ratio = height / width if width != 0 else 1

        # 🔥 Improved classification (YOLO + geometry)
        if cls == 0 or ratio < 0.8:
            label = "FALL"
        else:
            label = "NORMAL"

        # Risk score
        risk = 2 if label == "FALL" else 0
        if height < 100:
            risk += 1

        # Color
        color = (0,0,255) if label == "FALL" else (0,255,0)

        # Draw box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        # Text
        text = f"{label} {conf:.2f} | R:{risk}"
        cv2.putText(img, text, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Save result
    cv2.imwrite(os.path.join(output_folder, img_name), img)

print("Done")


0: 896x960 2 falls, 64.3ms
Speed: 8.1ms preprocess, 64.3ms inference, 1.5ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 2 falls, 43.8ms
Speed: 5.8ms preprocess, 43.8ms inference, 1.4ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 2 falls, 67.0ms
Speed: 5.7ms preprocess, 67.0ms inference, 1.4ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 3 falls, 66.9ms
Speed: 5.4ms preprocess, 66.9ms inference, 1.4ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 3 falls, 67.1ms
Speed: 11.4ms preprocess, 67.1ms inference, 3.4ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 2 falls, 66.9ms
Speed: 5.4ms preprocess, 66.9ms inference, 1.4ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 2 falls, 66.8ms
Speed: 5.5ms preprocess, 66.8ms inference, 1.5ms postprocess per image at shape (1, 3, 896, 960)

0: 896x960 2 falls, 67.0ms
Speed: 7.4ms preprocess, 67.0ms inference, 1.6ms postprocess per image at shape (1, 3, 896, 960)

In [ ]:
import os

label_folder = "/content/drive/MyDrive/fall_detection_dataset/labels/train"

normal_images = 0
fall_images = 0

for file in os.listdir(label_folder):
    if not file.endswith(".txt"):
        continue

    has_normal = False
    has_fall = False

    with open(os.path.join(label_folder, file)) as f:
        for line in f:
            if line.startswith("1"):
                has_normal = True
            elif line.startswith("0"):
                has_fall = True

    if has_normal:
        normal_images += 1
    if has_fall:
        fall_images += 1

print("Images with NORMAL:", normal_images)
print("Images with FALL:", fall_images)

Images with NORMAL: 167
Images with FALL: 208


In [ ]:
import os

normal_folder = "/content/drive/MyDrive/fall_detection_dataset/normal_images"

normal_images = set(os.listdir(normal_folder))

mismatch = 0

for file in os.listdir(label_folder):
    if not file.endswith(".txt"):
        continue

    image_name = file.replace(".txt", ".jpg")

    if image_name not in normal_images:
        mismatch += 1

print("Total mismatches:", mismatch)

Total mismatches: 248


In [ ]:
match = 0

for file in os.listdir(label_folder):
    if not file.endswith(".txt"):
        continue

    image_name = file.replace(".txt", ".jpg")

    if image_name in normal_images:
        match += 1

print("Exact matches:", match)

Exact matches: 179


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.6 MB/s eta 0:00:00


In [ ]:
import os
import cv2
from ultralytics import YOLO

# Load model
model = YOLO("/content/drive/MyDrive/fall_detection_dataset/best.pt")

# Input & output folders
input_folder = "/content/drive/MyDrive/fall_detection_test"
output_folder = "/content/drive/MyDrive/fall_detection_test_outputs"

os.makedirs(output_folder, exist_ok=True)

# Loop through all videos
for video_name in os.listdir(input_folder):

    if not video_name.endswith(".mp4"):
        continue

    video_path = os.path.join(input_folder, video_name)
    cap = cv2.VideoCapture(video_path)

    prev_y = None

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Output video
    out_path = os.path.join(output_folder, "output_" + video_name)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    print(f"🎥 Processing: {video_name}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=0.25, iou=0.5)[0]

        for box in results.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            center_y = (y1 + y2) // 2

            if prev_y is not None:
                speed = abs(center_y - prev_y)
            else:
                speed = 0

            prev_y = center_y

            height_box = y2 - y1
            width_box = x2 - x1
            ratio = height_box / width_box if width_box != 0 else 1

            # Final logic
            if cls == 0 or ratio < 0.8:
                label = "FALL"
            else:
                label = "NORMAL"


            # Risk score
            if label == "FALL":
              if speed > 20:
                  risk = 3
              elif speed > 10:
                  risk = 2
              else:
                  risk = 1
            else:
              risk = 0

            color = (0,0,255) if label == "FALL" else (0,255,0)

            # Draw
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, f"{label} {conf:.2f} | R:{risk}",
                        (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        out.write(frame)

    cap.release()
    out.release()

print("✅ All videos processed!")

Streaming output truncated to the last 5000 lines.

0: 544x960 2 falls, 32.0ms
Speed: 17.0ms preprocess, 32.0ms inference, 1.7ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.3ms
Speed: 12.8ms preprocess, 26.3ms inference, 1.9ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.4ms
Speed: 9.1ms preprocess, 26.4ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.3ms
Speed: 7.3ms preprocess, 26.3ms inference, 1.9ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.3ms
Speed: 8.4ms preprocess, 26.3ms inference, 2.2ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 27.4ms
Speed: 6.7ms preprocess, 27.4ms inference, 2.2ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.8ms
Speed: 8.2ms preprocess, 26.8ms inference, 2.2ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 2 falls, 26.9ms
Speed: 12.4ms preprocess, 26.9ms inference, 